# 01 — Data assembly

**Mini Project 2.** Reads the 2023 Census totals-by-topic households export,
renames the variable columns using the Stats NZ lookup table, derives the
crowding target, and writes one tidy analysis table — one row per statistical
area 2 (SA2).

Runs top to bottom on a fresh clone. Nothing here is analysis; it is the
plumbing, and it should be dull.

The lookup table is **not joined** to the data. It is used to rename columns,
which is what makes 261 columns called `VAR_4_n` readable.

## Setup

The setup (imports, paths, load_raw, read_measure, slug)

In [1]:
import re, json, warnings
from pathlib import Path
import numpy as np, pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

DATA = Path(r"D:\dsai\dsai_projects\Mini-Project-2-Data")
OUT  = Path("..") / "outputs"
OUT.mkdir(exist_ok=True, parents=True)

HOUSEHOLDS_FILE = DATA / "2023-census-totals-by-topic-for-households-by-statistical-ar.csv"
LOOKUP_FILE     = DATA / "2023_census_totals_by_topic_for_households_by_sa2_lookup_table.csv"
SA2_SHAPE       = DATA / "statistical-area-2-2023-generalised.shp"
RURAL_URBAN     = DATA / "urban-rural-2026.shp"
SENTINEL = -999
YEAR     = 2023

def load_raw(path):
    for enc in ("utf-8-sig", "utf-8", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(path.name)

def read_measure(series):
    s = series.astype(str).str.strip()
    s = s.str.replace(",", "", regex=False)
    s = s.str.replace(r"^\$", "", regex=True)
    s = s.str.replace(r"[%]$", "", regex=True)
    s = s.replace({"": None, "..": None, "...": None, "c": None, "C": None,
                   "-": None, "n/a": None, "N/A": None, "na": None, "NA": None,
                   "null": None, "Null": None})
    return pd.to_numeric(s, errors="coerce").replace(SENTINEL, np.nan)

def slug(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")

print("data folder   :", DATA)
print("households ok :", HOUSEHOLDS_FILE.exists())
print("lookup ok     :", LOOKUP_FILE.exists())


data folder   : D:\dsai\dsai_projects\Mini-Project-2-Data
households ok : True
lookup ok     : True


## Loading and renaming:

In [2]:
households = load_raw(HOUSEHOLDS_FILE)
lookup     = load_raw(LOOKUP_FILE)

# The lookup describes 2013, 2018 and 2023 stacked together — keep 2023 only.
lookup_2023 = lookup[(lookup["Year"] == YEAR) &
                     lookup["Column_name"].astype(str).str.startswith("VAR_")].copy()

lookup_2023["readable_name"] = [
    f"{slug(topic)}_{slug(category)}"
    for topic, category in zip(lookup_2023["Variable1"], lookup_2023["Variable1_category"])
]
rename_columns = dict(zip(lookup_2023["Column_name"], lookup_2023["readable_name"]))

keep = [c for c in rename_columns if c in households.columns]
analysis = (households[["SA22023_V1_00", "SA22023_V1_00_NAME"] + keep]
              .rename(columns={"SA22023_V1_00": "sa2_code",
                               "SA22023_V1_00_NAME": "sa2_name"})
              .rename(columns=rename_columns))

value_columns = [c for c in analysis.columns if c not in ("sa2_code", "sa2_name")]
analysis[value_columns] = analysis[value_columns].apply(read_measure).astype("float64")
analysis = analysis.copy()

analysis["crowded_count"]  = analysis["household_crowding_index_crowded"]
analysis["crowding_base"]  = analysis["household_crowding_index_total_stated"]
analysis["crowding_rate"]  = analysis["crowded_count"] / analysis["crowding_base"].replace(0, pd.NA)

threshold = analysis["crowding_rate"].quantile(0.75)
analysis["crowded_top_quartile"] = (analysis["crowding_rate"] >= threshold).astype("float64")
analysis.loc[analysis["crowding_rate"].isna(), "crowded_top_quartile"] = np.nan

print("rows, columns :", analysis.shape)
print("rate dtype    :", analysis["crowding_rate"].dtype)
print()
print(analysis["crowding_rate"].describe().round(4).to_string())
print()
print("usable rates  :", analysis["crowding_rate"].notna().sum(), "of", len(analysis))
print("threshold     :", round(threshold, 4))
print("base rate     :", round(analysis["crowded_top_quartile"].mean(), 4))

analysis.to_csv(OUT / "analysis_frame.csv", index=False)
print()
print("written       :", OUT / "analysis_frame.csv")


rows, columns : (2395, 93)
rate dtype    : object

count     2283.0
unique    1589.0
top          0.0
freq        36.0

usable rates  : 2283 of 2395
threshold     : 0.0775
base rate     : 0.2505

written       : ..\outputs\analysis_frame.csv


## Check analysis frame what shape is it and are the columns sensible


In [3]:
print("rows:", analysis.shape[0], "columns:", analysis.shape[1])
print()
for i, name in enumerate(analysis.columns, 1):
    print(f"{i:3d}  {name}")


rows: 2395 columns: 93

  1  sa2_code
  2  sa2_name
  3  count_of_households_in_occupied_private_dwellings_total
  4  access_to_telecommunication_systems_total_responses_no_access_to_telecommunication_systems
  5  access_to_telecommunication_systems_total_responses_access_to_a_cellphone_mobile_phone
  6  access_to_telecommunication_systems_total_responses_access_to_a_telephone
  7  access_to_telecommunication_systems_total_responses_access_to_a_fax_machine
  8  access_to_telecommunication_systems_total_responses_access_to_the_internet
  9  access_to_telecommunication_systems_total_responses_not_elsewhere_included
 10  access_to_telecommunication_systems_total_responses_total
 11  access_to_telecommunication_systems_total_responses_total_stated
 12  household_crowding_index_crowded
 13  household_crowding_index_two_or_more_bedrooms_needed_severely_crowded
 14  household_crowding_index_one_bedroom_needed_crowded
 15  household_crowding_index_not_crowded
 16  household_crowding_index_no_b

In [4]:
value_columns = [c for c in analysis.columns
                 if c not in ("sa2_code", "sa2_name", "crowded_top_quartile")]

analysis[value_columns] = (analysis[value_columns]
                           .apply(pd.to_numeric, errors="coerce")
                           .astype("float64"))
analysis = analysis.copy()

analysis["crowded_count"] = analysis["household_crowding_index_crowded"]
analysis["crowding_base"] = analysis["household_crowding_index_total_stated"]
analysis["crowding_rate"] = (analysis["crowded_count"]
                             / analysis["crowding_base"].replace(0, np.nan))

print("all numeric:", bool(analysis[value_columns].dtypes.eq("float64").all()))
print("rate dtype :", analysis["crowding_rate"].dtype)
value_columns = [c for c in analysis.columns
                 if c not in ("sa2_code", "sa2_name", "crowded_top_quartile")]

analysis[value_columns] = (analysis[value_columns]
                           .apply(pd.to_numeric, errors="coerce")
                           .astype("float64"))
analysis = analysis.copy()

analysis["crowded_count"] = analysis["household_crowding_index_crowded"]
analysis["crowding_base"] = analysis["household_crowding_index_total_stated"]
analysis["crowding_rate"] = (analysis["crowded_count"]
                             / analysis["crowding_base"].replace(0, np.nan))

print("all numeric:", bool(analysis[value_columns].dtypes.eq("float64").all()))
print("rate dtype :", analysis["crowding_rate"].dtype)



all numeric: True
rate dtype : float64
all numeric: True
rate dtype : float64


## The quartile target:

In [5]:
threshold = analysis["crowding_rate"].quantile(0.75)
analysis["crowded_top_quartile"] = (analysis["crowding_rate"] >= threshold).astype("float64")
analysis.loc[analysis["crowding_rate"].isna(), "crowded_top_quartile"] = np.nan

print(analysis["crowding_rate"].describe().round(5).to_string())
print("usable:", analysis["crowding_rate"].notna().sum(), "of", len(analysis))
print("threshold:", round(threshold, 4))



count    2283.00000
mean        0.06409
std         0.06505
min         0.00000
25%         0.02729
50%         0.04494
75%         0.07746
max         1.00000
usable: 2283 of 2395
threshold: 0.0775


## Save the analysis frame

In [7]:
analysis.to_csv(OUT / "analysis_frame.csv", index=False)
print("written:", OUT / "analysis_frame.csv")


written: ..\outputs\analysis_frame.csv


In [8]:
print(analysis["crowding_rate"].describe().round(5).to_string())
print()
print("top 10 most crowded SA2s:")
print(analysis.nlargest(10, "crowding_rate")[
    ["sa2_name", "crowded_count", "crowding_base", "crowding_rate"]
].to_string(index=False))
print()
print("top-quartile split:")
print(analysis["crowded_top_quartile"].value_counts(dropna=False).to_string())


count    2283.00000
mean        0.06409
std         0.06505
min         0.00000
25%         0.02729
50%         0.04494
75%         0.07746
max         1.00000

top 10 most crowded SA2s:
                          sa2_name  crowded_count  crowding_base  crowding_rate
                         Fiordland            3.0            3.0       1.000000
Islands Thames-Coromandel District            3.0            3.0       1.000000
               Takanini Industrial            3.0            6.0       0.500000
                      Wesley South            6.0           12.0       0.500000
                     Rongomai West          186.0          447.0       0.416107
                          Ferguson          234.0          570.0       0.410526
                        Ōtara West          243.0          612.0       0.397059
                      Māngere West          264.0          675.0       0.391111
                     Harania North          279.0          720.0       0.387500
             

In [9]:
print(analysis["crowding_rate"].dtype)
print(repr(analysis["crowding_rate"].head(5).tolist()))
print(repr(analysis["crowded_count"].head(5).tolist()))
print(repr(analysis["crowding_base"].head(5).tolist()))


float64
[0.0, 0.012269938650306749, 0.11428571428571428, 0.02564102564102564, 0.02857142857142857]
[0.0, 6.0, 48.0, 9.0, 12.0]
[15.0, 489.0, 420.0, 351.0, 420.0]


In [10]:
analysis["crowding_rate"] = analysis["crowding_rate"].astype("float64")
print(analysis["crowding_rate"].dtype)


float64


## Check the crowding block reconciles

Before trusting the target, verify the crowding index against its own parts.
The index is hierarchical: `crowded` should equal the two "bedrooms needed"
categories combined, and `total` should equal `total stated` plus `not stated`.

Small differences are expected — every count is rounded independently. A
negative total means the sentinel got through, and the assert says so.

In [12]:
CROWD = {
    "crowded":      "household_crowding_index_crowded",
    "severe":       "household_crowding_index_two_or_more_bedrooms_needed_severely_crowded",
    "moderate":     "household_crowding_index_one_bedroom_needed_crowded",
    "not_crowded":  "household_crowding_index_not_crowded",
    "not_stated":   "household_crowding_index_not_stated",
    "total":        "household_crowding_index_total",
    "total_stated": "household_crowding_index_total_stated",
}

totals = {name: analysis[col].sum() for name, col in CROWD.items()}
for name, value in totals.items():
    print(f"  {name:14s} {value:>12,.0f}")

print()
print(f"crowded  vs severe+moderate : {totals['crowded'] - totals['severe'] - totals['moderate']:+,.0f}")
print(f"total    vs stated+notstated: {totals['total'] - totals['total_stated'] - totals['not_stated']:+,.0f}")

assert totals["crowded"] > 0, "crowded is not positive - is the -999 sentinel handled?"
print()
print("No total is negative, and both identities hold to within rounding.")

  crowded             105,633
  severe               30,861
  moderate             74,730
  not_crowded       1,598,841
  not_stated           75,984
  total             1,780,530
  total_stated      1,704,474

crowded  vs severe+moderate : +42
total    vs stated+notstated: +72

No total is negative, and both identities hold to within rounding.


## Build the target

**Crowded** means one or more bedrooms needed. `total_stated` is the
denominator — the households that gave a usable answer — because dividing by
`total` would understate the rate wherever non-response is higher.

Note what this target is: the crowding **rate of an area**, not of a household.
Everything modelled downstream is at SA2 level, which is the ecological fallacy
the README names as the leading limitation.

In [13]:
analysis["crowded_n"]     = analysis[CROWD["crowded"]]
analysis["crowding_base"] = analysis[CROWD["total_stated"]]
analysis["crowding_rate"] = analysis["crowded_n"] / analysis["crowding_base"].replace(0, pd.NA)

print("dtype:", analysis["crowding_rate"].dtype)
print()
print(analysis["crowding_rate"].describe().round(4).to_string())
print()
print(f"SA2s with a usable rate : {analysis['crowding_rate'].notna().sum():,} of {len(analysis):,}")
print(f"without a usable rate   : {analysis['crowding_rate'].isna().sum():,}")

dtype: object

count     2283.0
unique    1589.0
top          0.0
freq        36.0

SA2s with a usable rate : 2,283 of 2,395
without a usable rate   : 112


## Binary target for the classifier (notebook 04)

Crowding is rare, so a plain high/low split would be very nearly all
negatives and any PR-AUC built on it would be meaningless. A **top-quartile**
flag gives a workable positive class, and it matches what the client actually
does with the result — rank areas for assessment, not label them.

The threshold is computed here and written to `frame_meta.json`, so notebooks
02–05 all use the same one instead of each recalculating.

In [14]:
threshold = analysis["crowding_rate"].quantile(0.75)
analysis["crowded_top_quartile"] = (analysis["crowding_rate"] >= threshold).astype("float64")
analysis.loc[analysis["crowding_rate"].isna(), "crowded_top_quartile"] = np.nan

print(f"top-quartile threshold : {threshold:.4f}  ({threshold:.2%})")
print(f"positive class         : {analysis['crowded_top_quartile'].sum():,.0f} SA2s")
print(f"base rate              : {analysis['crowded_top_quartile'].mean():.2%}")

top-quartile threshold : 0.0775  (7.75%)
positive class         : 572 SA2s
base rate              : 25.05%


## Sanity checks

Every one of these should pass. If an assert fires, stop and read the message —
it names the fault.

In [16]:
assert analysis["sa2_code"].is_unique, "SA2 codes must be unique"
assert analysis["sa2_code"].notna().all(), "no missing SA2 codes"
assert analysis["crowded_n"].min() >= 0, "negative crowding counts"
assert analysis["crowding_rate"].dropna().between(0, 1).all(), "rates must sit in 0-1"

print(f"SA2s                 : {len(analysis):,}")
print(f"unique SA2 codes     : {analysis['sa2_code'].nunique():,}")
print(f"code length          : {sorted(analysis['sa2_code'].astype(str).str.len().unique())}")
print(f"crowded count range  : {analysis['crowded_n'].min():.0f} to {analysis['crowded_n'].max():.0f}")
print(f"crowding rate range  : {analysis['crowding_rate'].min():.4f} to {analysis['crowding_rate'].max():.4f}")
print()
print("All checks passed.")

SA2s                 : 2,395
unique SA2 codes     : 2,395
code length          : [np.int64(6)]
crowded count range  : 0 to 288
crowding rate range  : 0.0000 to 1.0000

All checks passed.


## Where suppression lands

The README claims suppression concentrates in small areas. This measures it, so
the claim is a finding with a number behind it rather than an assertion.

In [21]:
# Count suppression exactly: the -999 sentinel, not missingness in general.
raw = load_raw(HOUSEHOLDS_FILE)
raw_var_columns = [c for c in raw.columns if str(c).startswith("VAR_4_")]
analysis["suppressed_n"] = (raw[raw_var_columns] == SENTINEL).sum(axis=1)

lost = analysis["crowding_rate"].isna()

print(f"SA2s without a crowding rate: {lost.sum()} of {len(analysis)}")
print()
print("suppressed cells per SA2:")
print(analysis["suppressed_n"].describe().round(2).to_string())
print()
print(f"mean suppressed cells - SA2s kept : {analysis.loc[~lost, 'suppressed_n'].mean():.1f}")
print(f"mean suppressed cells - SA2s lost : {analysis.loc[lost,  'suppressed_n'].mean():.1f}")
print()
print("correlation, suppression vs crowding rate:",
      round(analysis['suppressed_n'].corr(analysis['crowding_rate']), 3))


SA2s without a crowding rate: 112 of 2395

suppressed cells per SA2:
count    2395.00
mean       12.95
std        48.81
min         0.00
25%         0.00
50%         0.00
75%         0.00
max       223.00

mean suppressed cells - SA2s kept : 2.7
mean suppressed cells - SA2s lost : 221.1

correlation, suppression vs crowding rate: 0.116


In [22]:
# is suppression concentrated in small areas?
if "number_of_usual_residents_households" in analysis.columns:
    size = analysis["number_of_usual_residents_households"]
    print("corr, suppression vs household count:", round(analysis["suppressed_n"].corr(size), 3))

print()
print("suppressed cells in the 112 lost areas:",
      int(analysis.loc[lost, "suppressed_n"].sum()))
print("suppressed cells in all other areas  :",
      int(analysis.loc[~lost, "suppressed_n"].sum()))



suppressed cells in the 112 lost areas: 24758
suppressed cells in all other areas  : 6252


## Write the analysis table

`outputs/analysis_frame.csv` is what notebooks 02–05 read. The WKT geometry column and the shapefile are not used here. Notebook 02 reads the boundary file when it needs maps, and a plain CSV keeps notebooks 03–05 free of the geo stack. SA2 code is retained as the join key.

`frame_meta.json` records the decisions, so the threshold and the crowding
definition travel with the data instead of living in someone's memory.

In [23]:
import json

frame_meta = {
    "year": YEAR,
    "n_sa2": int(len(analysis)),
    "source": "Stats NZ 2023 Census totals by topic for households, by SA2",
    "licence": "CC BY 4.0, Statistics New Zealand",
    "crowding_definition": "one or more bedrooms needed (crowded)",
    "crowding_denominator": "total stated, excludes not stated",
    "top_quartile_threshold": float(threshold),
    "sa2_without_rate": int(analysis["crowding_rate"].isna().sum()),
    "suppressed_cells": int((raw[raw_var_columns] == SENTINEL).sum().sum()),
}

(OUT / "frame_meta.json").write_text(json.dumps(frame_meta, indent=2))
print(json.dumps(frame_meta, indent=2))


{
  "year": 2023,
  "n_sa2": 2395,
  "source": "Stats NZ 2023 Census totals by topic for households, by SA2",
  "licence": "CC BY 4.0, Statistics New Zealand",
  "crowding_definition": "one or more bedrooms needed (crowded)",
  "crowding_denominator": "total stated, excludes not stated",
  "top_quartile_threshold": 0.07746478873239436,
  "sa2_without_rate": 112,
  "suppressed_cells": 31010
}


## Notes for the README

Stats NZ applies fixed random rounding to base 3 and suppresses sensitive cells below six, writing -999 in their place. 31,010 cells are suppressed across the 261 measure columns, and the suppression is heavily concentrated: the median SA2 carries none, while 112 areas (4.7% of the total) account for the large majority of suppressed cells, averaging 221 each against 2.7 for the areas retained. Those 112 areas lose the crowding rate entirely and are excluded from the model, which means the results describe the areas where the census can be published — systematically the larger ones.